In [9]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round

In [1]:
import os
import pyspark

spark_version = pyspark.__version__
print(f"Detected PySpark version: {spark_version}")

if spark_version.startswith("4"):
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
else:
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

print(f"Using connector: {KAFKA_PACKAGE}")

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {KAFKA_PACKAGE} pyspark-shell"

Detected PySpark version: 4.0.0.dev2
Using connector: org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} - ready")

Spark 4.0.0-preview2 - ready


In [3]:
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "host.docker.internal:29092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .load()
)

kafka_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
from pyspark.sql.functions import col

batch_counter = {"n": 0}

def peek_raw(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id}: {df.count()} messages ---")
    df.select(
        "topic",
        "partition",
        "offset",
        "timestamp",
        col("key").cast("string").alias("key"),
        col("value").cast("string").alias("value"),
    ).show(5, truncate=100)

    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = (
    kafka_raw.writeStream
    .foreachBatch(peek_raw)
    .option("checkpointLocation", "/tmp/chk_lab4_peek")
    .start()
)

try:
    q.awaitTermination()
except:
    q.stop()

In [6]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json, to_timestamp

tx_schema = StructType([
    StructField("tx_id", StringType()),
    StructField("user_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("store", StringType()),
    StructField("category", StringType()),
    StructField("timestamp", StringType()),
])

df = (
    kafka_raw
    .select(from_json(col("value").cast("string"), tx_schema).alias("tx"))
    .select("tx.*")
    .withColumn("timestamp", to_timestamp("timestamp"))
)

df.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- store: string (nullable = true)
 |-- category: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [7]:
batch_counter["n"] = 0

def show_parsed(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id} ---")
    df.show(5, truncate=False)

    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = (
    df.writeStream
    .foreachBatch(show_parsed)
    .option("checkpointLocation", "/tmp/chk_lab4_parsed")
    .start()
)

try:
    q.awaitTermination()
except:
    q.stop()

In [ ]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round

windowed = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "1 minute"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount")
    )
)

batch_counter["n"] = 0

def show_window(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} ===")
    (
        df.select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "store",
            "tx_count",
            "total_amount"
        )
        .orderBy("from", "store")
        .show(truncate=False)
    )

    if batch_counter["n"] >= 5:
        raise Exception("stop")

q = (
    windowed.writeStream
    .outputMode("append")
    .foreachBatch(show_window)
    .option("checkpointLocation", "/tmp/chk_lab4_windows")
    .start()
)

try:
    q.awaitTermination()
except:
    q.stop()

In [10]:
windowed_cat = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "1 minute"), "category")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount")
    )
)

batch_counter["n"] = 0

def show_window_cat(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} ===")
    (
        df.select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "category",
            "tx_count",
            "total_amount"
        )
        .orderBy("from", "category")
        .show(truncate=False)
    )

    if batch_counter["n"] >= 5:
        raise Exception("stop")

q = (
    windowed_cat.writeStream
    .outputMode("complete")
    .foreachBatch(show_window_cat)
    .option("checkpointLocation", "/tmp/chk_lab4_cat_complete")
    .start()
)

try:
    q.awaitTermination()
except:
    q.stop()

In [11]:
from pyspark.sql.functions import to_json, struct, lit

alerts = (
    df
    .filter(col("amount") > 3000)
    .select(
        to_json(
            struct(
                "tx_id",
                "user_id",
                "amount",
                "store",
                "category",
                col("timestamp").cast("string"),
                lit("HIGH").alias("alert_level")
            )
        ).alias("value")
    )
)

alert_query = (
    alerts.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "host.docker.internal:29092")
    .option("topic", "alerts")
    .option("checkpointLocation", "/tmp/chk_lab4_alerts")
    .outputMode("append")
    .start()
)

print("Alert stream started")

Alert stream started


In [12]:
sliding_store = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "2 minutes", "1 minute"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount")
    )
)

batch_counter["n"] = 0

def show_sliding_store(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} ===")
    (
        df.select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "store",
            "tx_count",
            "total_amount"
        )
        .orderBy("from", "store")
        .show(truncate=False)
    )

    if batch_counter["n"] >= 5:
        raise Exception("stop")

q = (
    sliding_store.writeStream
    .outputMode("complete")
    .foreachBatch(show_sliding_store)
    .option("checkpointLocation", "/tmp/chk_lab4_hw_sliding_store")
    .start()
)

try:
    q.awaitTermination()
except:
    q.stop()

In [13]:
alerts_ratio = (
    df
    .filter(col("amount") > 3000)
    .withColumn("ratio", col("amount") / 4000.0)
    .select(
        to_json(
            struct(
                "tx_id",
                "user_id",
                "amount",
                "ratio",
                "store",
                "category",
                col("timestamp").cast("string"),
                lit("HIGH").alias("alert_level")
            )
        ).alias("value")
    )
)

alert_ratio_query = (
    alerts_ratio.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "host.docker.internal:29092")
    .option("topic", "alerts")
    .option("checkpointLocation", "/tmp/chk_lab4_alerts_ratio")
    .outputMode("append")
    .start()
)

print("Alert ratio stream started")

Alert ratio stream started


In [14]:
# Review questions

# 1. Append mode has a delay because Spark waits until the window is closed
#    and the watermark confirms that late data is unlikely.

# 2. If watermark is set to 0 seconds, Spark accepts almost no late data.
#    Late events are more likely to be dropped.

# 3. In Lab 2, window() was applied to static batch data.
#    Here, window() is applied to streaming data, so results update over time
#    and depend on output mode and watermark.

# 4. Kafka expects one value column as bytes/string.
#    to_json() converts structured columns into one JSON string for Kafka.

# Homework 3:
# If the producer is stopped and we wait 2 minutes, no new Kafka messages arrive.
# The streaming query keeps running, but there are no new rows.
# Windowed results may stop updating because the watermark also stops advancing
# without new event-time data.

In [15]:
try:
    alert_query.stop()
except:
    pass

try:
    alert_ratio_query.stop()
except:
    pass

spark.stop()